In [ ]:
# Colab setup -- installs SoftMobility when running on Google Colab.
# Safe to run locally: it does nothing outside Colab.
try:
    import google.colab  # noqa: F401
    %pip install -q git+https://github.com/C0PEP0D/SoftMobility.git
except ImportError:
    pass

# Tutorial 13 — Actuated filaments (Figs 12 and 13 of Delmotte et al. 2015)

Where tutorial 12 followed *passive* dynamics of a flexible fiber in
external shear, this notebook reproduces two **actuated** filament
configurations from §3.4 of Delmotte, Climent & Plouraboué 2015
(*J. Comput. Phys.* **286**, 14–37):

1. **Fig. 12** — A flexible filament tethered at one end and oscillated
   in the plane (Yu, Lauga & Hosoi 2006).
2. **Fig. 13 inset** — Helical actuation of a tethered filament (Coq,
   du Roure, Marthelot, Bartolo & Fermigier 2008).

Both rely on the time-varying **clamping API** of
`FlowBodyRollout.rollout`. The user supplies a callable for the body
position, body orientation, and/or specific DOFs as a function of time;
after each integrator step the corresponding components are overwritten
with the prescribed values, while the unclamped components evolve under
the bending dynamics.

**Linearization caveat.** As in tutorial 12, the bending torque is
linearized around the straight configuration. The amplitude of the
prescribed actuation must therefore be kept modest (joint angles
≲ π/4, in this notebook ≲ 0.3 rad) for the linearization to remain
quantitatively useful. With the prescribed amplitude held in that
regime, the tail of the filament beyond the clamp develops a wave
profile that is qualitatively similar to the paper's Figs. 12 and 13.

In [1]:
import jax.numpy as jnp
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import softmobility as sm

np.set_printoptions(precision=3, suppress=True)

## Helper — lab-frame bead positions

Same helper as tutorial 12: convert the body-reference rollout output
into lab-frame bead positions for plotting.

In [2]:
def lab_bead_positions(fiber, positions, orientations, dofs, design=None):
    """Return an array of shape (n_steps, n_beads, 3) with lab-frame bead positions."""
    if design is None:
        design = np.asarray(fiber.design_defaults)
    n_steps = int(positions.shape[0])
    n_beads = fiber.Nspheres
    t_dummy = np.array([0.0])
    out = np.zeros((n_steps, n_beads, 3))
    body_pos = np.zeros((n_beads, 3))
    for step in range(n_steps):
        dof_step = np.asarray(dofs[step])
        for i in range(n_beads):
            body_pos[i] = np.asarray(fiber.spheres[i].position(dof_step, design, t_dummy))
        R = np.asarray(sm.rotation_matrix(orientations[step]))
        out[step] = np.asarray(positions[step]) + body_pos @ R.T
    return out

## Part 1 — Planar oscillation (Fig. 12)

Following Yu, Lauga & Hosoi 2006 and §3.4.1.1 of the paper: a planar
filament has its first bead pinned at the origin, and the second bead is
forced on the prescribed arc

$$ \mathbf{r}_2(t) \;=\; 2a\,\bigl(\cos(\alpha_0\sin\zeta t),\;0,\;\sin(\alpha_0\sin\zeta t)\bigr). $$

In our implicit-DOF parameterization the chain anchor (bead 0) coincides
with the body's frame origin and the second bead's lab position is
$2a\cos(\theta_1/2)\,\bigl(\cos(\theta_1/2),\,0,\,\sin(\theta_1/2)\bigr)$
for $\theta_0=0$. Setting the bond direction angle $\theta_1/2$ equal to
$\beta(t) = \alpha_0\sin\zeta t$ gives $\theta_1(t) = 2\beta(t)$. The
required clamps are therefore:

* `body_position = (0, 0, 0)` (constant)
* `body_orientation = (0, 0, 0)` (constant — no rotation of the body
  frame, so lab and body frames coincide)
* `θ_0 = 0` (constant)
* `θ_1(t) = 2 α_0 sin(ζ t)`

The remaining DOFs (`θ_2 … θ_{N-1}`) evolve under the bending dynamics.

In [3]:
N = 20
a = 1.0
mu = 1.0
alpha0 = 0.3        # ~ 17° actuation amplitude (kept inside linearization range)
zeta = 1.0          # actuation angular frequency
K_b_planar = 50.0   # bending modulus

fiber = sm.FlexibleFiber(n_beads=N, radius=a, bending_rigidity=K_b_planar, mass=0.0, planar=True)
rollout = sm.FlowBodyRollout(
    soft_body=fiber,
    flow=sm.no_flow(),
    input_map={"gravity": sm.gravity_field(g=0.0)},
)

# Rough Sperm number estimate (RFT slender approximation, γ⊥/L ≈ 4πμ):
L_planar = (N - 1) * 2 * a
Sp = L_planar * (zeta * 4 * np.pi * mu / L_planar / K_b_planar) ** 0.25
print(f"L = {L_planar:.1f}, K_b = {K_b_planar:.1f}, Sp ≈ {Sp:.2f}")

# Build the clamps. Mask: pin θ_0 and θ_1.
clamp_mask_planar = jnp.zeros(fiber.Ndof, dtype=bool).at[0].set(True).at[1].set(True)


def clamp_dofs_planar(t):
    return jnp.zeros(fiber.Ndof).at[1].set(2.0 * alpha0 * jnp.sin(zeta * t))


def clamp_position_origin(t):
    return jnp.zeros(3)


def clamp_orientation_zero(t):
    return jnp.zeros(3)

L = 38.0, K_b = 50.0, Sp ≈ 10.84


### Run the rollout

We integrate over a few actuation periods so that any transient from the
straight initial condition has died out. Time step is set by the bending
time scale `μ(2a)^4 / K_b`.

In [4]:
period = 2 * np.pi / zeta
dt_planar = 0.05 * 16 * mu / K_b_planar  # ~ 5 % of the bending time scale
n_steps_planar = int(np.ceil(3 * period / dt_planar))  # 3 actuation periods
print(f"period = {period:.2f}, dt = {dt_planar:.4f}, n_steps = {n_steps_planar}")

positions_p, orientations_p, dofs_p = rollout.rollout(
    dt=dt_planar,
    n_steps=n_steps_planar,
    clamp_position_fn=clamp_position_origin,
    clamp_orientation_fn=clamp_orientation_zero,
    clamp_dofs_mask=clamp_mask_planar,
    clamp_dofs_fn=clamp_dofs_planar,
)
positions_p.block_until_ready()
lab_p = lab_bead_positions(fiber, positions_p, orientations_p, dofs_p)
print(f"shape: {lab_p.shape}, max bead z: {np.max(np.abs(lab_p[:, :, 2])):.3f}")

period = 6.28, dt = 0.0160, n_steps = 1179
shape: (1179, 20, 3), max bead z: 1.659


### Plot — fiber shape across one actuation period (Fig. 12 layout)

Snapshots equally spaced over the **last** actuation period (after the
initial transient), in the lab frame. The proximal bead is pinned at the
origin and the actuated bead 1 traces an arc of radius ``2a cos(α₀)``.
The remainder of the fiber follows by elastic propagation; the wave
profile depends on the Sperm number.

In [5]:
# Take 8 snapshots from the last actuation period
period_steps = int(period / dt_planar)
start = n_steps_planar - period_steps
snap_idx = np.linspace(start, n_steps_planar - 1, 8, dtype=int)

fig = go.Figure()
greys = [f"rgb({g},{g},{g})" for g in np.linspace(20, 200, len(snap_idx)).astype(int)]
for c, k in zip(greys, snap_idx, strict=False):
    snap = lab_p[k]
    fig.add_trace(
        go.Scatter(
            x=snap[:, 0],
            y=snap[:, 2],
            mode="lines+markers",
            name=f"t = {k * dt_planar:.2f}",
            line=dict(color=c, width=2),
            marker=dict(size=5, color=c),
        )
    )

fig.update_layout(
    title=f"Planar oscillation (Sp ≈ {Sp:.2f}, α₀ = {alpha0}, N = {N})",
    xaxis_title="x",
    yaxis_title="z",
    yaxis=dict(scaleanchor="x", scaleratio=1),
    width=750,
    height=500,
    plot_bgcolor="white",
)
fig.show()

## Part 2 — Helical actuation (Fig. 13 inset)

In the helical case (§3.4.1.2; Coq et al. 2008) the proximal bead is
attached to a small platform that rotates around a fixed axis at radius
``δ̃₀ sin α₀``, with the bead's tangent pointing slightly off-axis by
α₀. As the platform rotates, the rest of the fiber whips behind it and
develops a 3-D helical-like shape.

We approximate this with a **3-D fiber** (planar = False), clamp the
body's lab position on a small circle in the y-z plane, and clamp the
body Rodrigues vector to a constant rotation that tilts the body's
``ê_x`` (= initial tangent of the chain) off the rotation axis by α₀.
The proximal two beads' rod DOFs are also clamped so the chain enters
the actuator in a fixed orientation; the rest follows elastically.

In [6]:
N3 = 12
alpha0_h = 0.25      # ~ 14° tilt off the rotation axis
zeta_h = 1.0
delta0 = 2.7 * a     # platform offset (paper Eq. 62 with δ̃₀)
K_b_helical = 30.0

fiber3 = sm.FlexibleFiber(n_beads=N3, radius=a, bending_rigidity=K_b_helical, mass=0.0, planar=False)
rollout3 = sm.FlowBodyRollout(
    soft_body=fiber3,
    flow=sm.no_flow(),
    input_map={"gravity": sm.gravity_field(g=0.0)},
)
L_helical = (N3 - 1) * 2 * a
Sp_h = L_helical * (zeta_h * 4 * np.pi * mu / L_helical / K_b_helical) ** 0.25
print(f"L = {L_helical:.1f}, K_b = {K_b_helical:.1f}, Sp ≈ {Sp_h:.2f}")


def helical_position(t):
    """Anchor on a circle in the y-z plane at offset δ̃₀ sin α₀."""
    radius = delta0 * jnp.sin(alpha0_h)
    return jnp.array([0.0, radius * jnp.cos(zeta_h * t), radius * jnp.sin(zeta_h * t)])


def helical_orientation(t):
    """Rotate body so its ê_x makes angle α₀ with the rotation axis (ê_x lab)
    in the plane containing the platform position. The Rodrigues rotation
    axis lies along (ê_x × r̂_platform), and the angle is α₀."""
    radius_xy = jnp.array([0.0, jnp.cos(zeta_h * t), jnp.sin(zeta_h * t)])
    # rotation that maps lab ê_x onto the slightly-tilted body ê_x
    axis = jnp.cross(jnp.array([1.0, 0.0, 0.0]), radius_xy)
    axis = axis / (jnp.linalg.norm(axis) + 1e-12)
    return alpha0_h * axis


# Clamp bead 0 and bead 1 rod DOFs to zero (chain enters straight along body ê_x).
clamp_mask_h = jnp.zeros(fiber3.Ndof, dtype=bool).at[0:6].set(True)


def clamp_dofs_h(t):
    return jnp.zeros(fiber3.Ndof)

L = 22.0, K_b = 30.0, Sp ≈ 8.17


In [7]:
period_h = 2 * np.pi / zeta_h
dt_h = 0.05 * 16 * mu / K_b_helical
n_steps_h = int(np.ceil(3 * period_h / dt_h))
print(f"period = {period_h:.2f}, dt = {dt_h:.4f}, n_steps = {n_steps_h}")

positions_h, orientations_h, dofs_h = rollout3.rollout(
    dt=dt_h,
    n_steps=n_steps_h,
    init_position=helical_position(0.0),
    init_orientation=helical_orientation(0.0),
    clamp_position_fn=helical_position,
    clamp_orientation_fn=helical_orientation,
    clamp_dofs_mask=clamp_mask_h,
    clamp_dofs_fn=clamp_dofs_h,
)
positions_h.block_until_ready()
lab_h = lab_bead_positions(fiber3, positions_h, orientations_h, dofs_h)
print(f"shape: {lab_h.shape}")

period = 6.28, dt = 0.0267, n_steps = 707
shape: (707, 12, 3)


### Plot — 3-D shape over the last actuation period

Twenty equally-spaced snapshots in the last actuation period. Lighter
shades correspond to later times within the period. The proximal end
traces the small circle of radius δ̃₀ sin α₀ around the rotation axis,
and the rest of the fiber whips in the rotating frame.

In [8]:
period_steps_h = int(period_h / dt_h)
start_h = n_steps_h - period_steps_h
snap_idx_h = np.linspace(start_h, n_steps_h - 1, 20, dtype=int)

fig = go.Figure()
greys_h = [f"rgb({g},{g},{g})" for g in np.linspace(20, 200, len(snap_idx_h)).astype(int)]
for c, k in zip(greys_h, snap_idx_h, strict=False):
    snap = lab_h[k]
    fig.add_trace(
        go.Scatter3d(
            x=snap[:, 0], y=snap[:, 1], z=snap[:, 2],
            mode="lines+markers",
            line=dict(color=c, width=4),
            marker=dict(size=3, color=c),
            showlegend=False,
        )
    )

fig.update_layout(
    title=f"Helical actuation (Sp ≈ {Sp_h:.2f}, α₀ = {alpha0_h}, N = {N3})",
    scene=dict(
        xaxis_title="x", yaxis_title="y", zaxis_title="z",
        aspectmode="data",
    ),
    width=750,
    height=600,
)
fig.show()

## Notes

* The clamping API is **post-step**: at every integrator step the body
  position, orientation, and the masked DOFs are overwritten with the
  prescribed values. The unclamped DOFs evolve dynamically under the
  hydrodynamic and bending forces produced *with the clamps in place*,
  which is what makes the rest of the fiber respond elastically to the
  prescribed actuation.
* In the planar case (Part 1) we exploited that bead 0 sits at the body
  origin; pinning the body position pins bead 0. Pinning ``θ_0 = 0``
  fixes the chain's first tangent along ``ê_x``, and prescribing
  ``θ_1(t)`` directly controls the bond from bead 0 to bead 1. Bond
  *length* in our parameterization is ``2a·cos(θ_1/2)``, so the
  prescribed actuation is geometrically exact only at small α₀; we keep
  α₀ ≤ 0.3 here, consistent with the bending linearization.
* The helical case (Part 2) is a simplification of the paper's setup:
  we prescribe both the body position and the body orientation (a
  smooth rotation that tracks the platform) and pin the first two
  beads' rod DOFs. The paper's full prescription (Eq. 62–64) constrains
  bead 1's lab position too; in our parameterization that would require
  solving for ``rod_1`` consistent with the desired bead 1 position
  given the body state — straightforward but algebraically involved
  and unnecessary for the qualitative wave profile shown here.
* Sperm number ``Sp = L · (ζ γ⊥/L / K_b)^(1/4)`` is reported using the
  RFT slender approximation ``γ⊥/L ≈ 4πμ`` and is therefore an
  order-of-magnitude estimate. For quantitative comparison with the
  paper one would compute γ⊥ from the full mobility tensor of a
  straight rigid chain.